[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/lines_circles_course.ipynb)

# Coordinate geometry full course: straight lines and circles, practice notebook

One practice question for every teaching chapter of the video, in the same order, and a plane, a line and a circle
you can move yourself. Work each question out on paper first, then type your answer into the box and press
**check**. It tells you whether you're right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Fractions as `2/3`, roots as `sqrt(34)`, minus as `-`. Angles in degrees as plain numbers
(`45`). For several answers, separate them with commas, in the order the question asks. When a question asks for an
equation, type it in any form you like, such as `y = 2x + 1` or `2x - y + 1 = 0`: the check knows they're the same
line.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re, textwrap
from fractions import Fraction as F
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD = "#6fb6bb", "#a0c27c", "#e8c56b"     # across · drops and distances · the answer

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x, y = sp.symbols("x y")
r = sp.Rational
NAMES = {"x": x, "y": y, "pi": sp.pi, "sqrt": sp.sqrt, "I": 1, "II": 2, "III": 3, "IV": 4}   # quadrants as I–IV too

def read_one(p):
    """One typed answer: a number, or an equation (everything moved to one side, so it equals zero)."""
    if "=" in p:
        left, right = p.replace("==", "=").split("=", 1)
        return sp.expand(parse_expr(left, local_dict=NAMES, transformations=RULES)
                         - parse_expr(right, local_dict=NAMES, transformations=RULES))
    return parse_expr(p, local_dict=NAMES, transformations=RULES)

def prep(s):
    """Typed symbols → Python: − × ÷ ° ² √."""
    s = s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("°", "").replace("²", "^2")
    return re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", s).replace("√", "sqrt")   # √34 → sqrt(34); √(…) stays

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [read_one(p) for p in prep(s).split(",") if p.strip()]

def same_curve(got, want):
    """Two equations (each as 'something = 0') describe the same curve: one is a number times the other."""
    got, want = sp.expand(sp.sympify(got)), sp.expand(want)
    if got == 0 or not got.free_symbols <= {x, y}: return False
    mono, cw = sp.Poly(want, x, y).terms()[0]
    cg = sp.Poly(got, x, y).coeff_monomial(mono)
    return cg != 0 and sp.expand(got * cw - want * cg) == 0

def same(got, want, tol=1e-9):
    if sp.sympify(want).free_symbols: return same_curve(got, want)          # an equation
    try: return abs(complex(sp.N(sp.sympify(got) - want, 30))) < tol
    except TypeError: return False                                         # a letter, not a number

def fr(v):
    """Any exact number (Python or sympy) as a Fraction."""
    return F(int(v.p), int(v.q)) if hasattr(v, "q") else F(v).limit_denominator(10 ** 6)

def num(v):
    """A fraction the way you'd write it: 2/3, −1/2, 5."""
    v = fr(v)
    s = f"{v.numerator}" if v.denominator == 1 else f"{v.numerator}/{v.denominator}"
    return s.replace("-", "−")

def dist(P, R): return math.sqrt((R[0] - P[0]) ** 2 + (R[1] - P[1]) ** 2)    # the distance formula

def grid(ax, lim=6):
    """The plane: x and y axes through the origin, one unit the same both ways, ticks at every whole number."""
    ax.set_xlim(-lim - 0.5, lim + 0.5); ax.set_ylim(-lim - 0.5, lim + 0.5); ax.set_aspect("equal")
    ax.set_xticks(range(-lim, lim + 1, 2)); ax.set_yticks(range(-lim, lim + 1, 2))
    ax.axhline(0, color=DIM, lw=1.4); ax.axvline(0, color=DIM, lw=1.4)
    ax.text(lim + 0.4, 0.25, "x", color=DIM, ha="right"); ax.text(0.25, lim + 0.2, "y", color=DIM, va="top")

def quadrant(px, py):
    """Which quadrant a point is in, or which axis it sits on."""
    if px == 0 and py == 0: return "the origin"
    if py == 0: return "on the x axis: no quadrant"
    if px == 0: return "on the y axis: no quadrant"
    return {(1, 1): "quadrant I", (-1, 1): "quadrant II", (-1, -1): "quadrant III", (1, -1): "quadrant IV"}[
        (1 if px > 0 else -1, 1 if py > 0 else -1)]

# Each question: the question, the answers in order, the working, (for rounded answers) how close counts, (if it has
# several parts) the move that cracks each part, shown when that part is wrong, and (sometimes) a picture of the
# working.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            tol = q.get("tol", [1e-9] * len(want))
            ok = [same(g, w, t) for g, w, t in zip(got, want, tol)]
            if all(ok): print("✓ right"); return
            if "moves" not in q: print("✗ not quite: try again, or press show working"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            if "picture" in q: q["picture"](q["working"]); return     # the working drawn beside its picture
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

## 1 · The coordinate plane

Two number lines cross at right angles at the origin O(0, 0): the x axis across, the y axis up. Every point is an
ordered pair (x, y): how far across, then how far up. The axes cut the plane into four quadrants, numbered I to IV
anticlockwise from the top right: (+, +), (−, +), (−, −), (+, −). A point on an axis is in no quadrant. Move the two
sliders: the point moves, and its coordinates and its quadrant update as it goes.

In [ ]:
SIGNS = {"quadrant I": "(+, +)", "quadrant II": "(−, +)", "quadrant III": "(−, −)", "quadrant IV": "(+, −)"}
assert [quadrant(*P) for P in ((-2, 3), (-4, -1), (5, -2), (3, 4), (0, 3), (-5, 0))] == \
       ["quadrant II", "quadrant III", "quadrant IV", "quadrant I", "on the y axis: no quadrant",
        "on the x axis: no quadrant"]                                      # the video's points

def plane(px=3, py=2):
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.02, 0.06, 0.5, 0.88]); grid(ax)
    for lab, (qx, qy) in zip(("I", "II", "III", "IV"), ((4.5, 4.5), (-4.5, 4.5), (-4.5, -4.5), (4.5, -4.5))):
        here = quadrant(px, py) == f"quadrant {lab}"
        ax.text(qx, qy, lab, color=CORAL if here else GHOST, fontsize=26 if here else 20, ha="center", va="center",
                fontweight="bold")
    ax.plot([px, px], [0, py], color=MINT, lw=2, ls="--"); ax.plot([0, px], [py, py], color=MINT, lw=2, ls="--")
    ax.plot([0, px], [0, 0], color=SKY, lw=4, solid_capstyle="butt")                     # across
    ax.plot([px], [py], "o", color=CORAL, ms=14, zorder=6)
    ax.text(px + (0.35 if px >= 0 else -0.35), py + 0.45, f"({num(px)}, {num(py)})", color=CORAL, fontsize=16,
            ha="left" if px >= 0 else "right", fontweight="bold")
    p = fig.add_axes([0.56, 0.0, 0.44, 1.0]); p.axis("off")
    p.text(0.02, 0.86, "the point", color=DIM, fontsize=14)
    p.text(0.02, 0.70, f"({num(px)}, {num(py)})", color=CORAL, fontsize=34, family="monospace", fontweight="bold")
    across = f"{abs(px)} right" if px > 0 else f"{abs(px)} left" if px < 0 else "no move across"
    up = f"{abs(py)} up" if py > 0 else f"{abs(py)} down" if py < 0 else "no move up"
    p.text(0.02, 0.56, f"x = {num(px)}: {across}\ny = {num(py)}: {up}", color=CHALK, fontsize=15, va="top",
           linespacing=1.6)
    where = quadrant(px, py)
    p.text(0.02, 0.26, where, color=GOLD, fontsize=24, fontweight="bold")
    p.text(0.02, 0.14, f"signs {SIGNS[where]}" if where in SIGNS else "zero on an axis", color=DIM, fontsize=15)
    plt.show()

PTS1 = {"A": (-4, 2), "B": (3, -5), "C": (-2, -3)}          # question 1's points

def picture1(working):
    """Question 1's working: the three points plotted, the steps beside them."""
    fig = plt.figure(figsize=(11.5, 3.6))
    ax = fig.add_axes([0.0, 0.05, 0.3, 0.9]); grid(ax); ax.set_xticks([]); ax.set_yticks([])
    for name, (px, py) in PTS1.items():
        ax.plot([px, px], [0, py], color=MINT, lw=1.6, ls="--"); ax.plot([0, px], [py, py], color=MINT, lw=1.6, ls="--")
        ax.plot([px], [py], "o", color=CORAL, ms=10)
        ax.text(px + (0.4 if px > 0 else -0.4), py, name, color=CORAL, fontsize=15, va="center",
                ha="left" if px > 0 else "right", fontweight="bold")
    t = fig.add_axes([0.32, 0.0, 0.68, 1.0]); t.axis("off")
    for k, line in enumerate(working): t.text(0.0, 0.78 - 0.24 * k, line, color=CHALK, fontsize=16)
    plt.show()

In [ ]:
interact(plane, px=IntSlider(value=3, min=-6, max=6, description="x"),
         py=IntSlider(value=2, min=-6, max=6, description="y"));

In [ ]:
Q[1] = {"text": "Plot A(−4, 2), B(3, −5) and C(−2, −3) on paper. Then name the quadrant each one is in, in the order "
                "A, B, C (type 2 or II for the second quadrant).",
        "answer": [2, 4, 3],
        "moves": ["A: x is negative and y is positive.", "B: x is positive and y is negative.",
                  "C: both numbers are negative."],
        "working": ["A(−4, 2): 4 left, then 2 up. Signs (−, +): quadrant II.",
                    "B(3, −5): 3 right, then 5 down. Signs (+, −): quadrant IV.",
                    "C(−2, −3): 2 left, then 3 down. Signs (−, −): quadrant III."],
        "picture": picture1}
assert [quadrant(*P) for P in PTS1.values()] == ["quadrant II", "quadrant IV", "quadrant III"]
practice(1)

## 2 · The distance formula

Join two points with a slanting line, and draw the right-angled triangle under it: across x₂ − x₁, up y₂ − y₁. By
Pythagoras' theorem the distance is d = √((x₂ − x₁)² + (y₂ − y₁)²). The order doesn't matter, because squaring
removes the sign.

In [ ]:
Q[2] = {"text": "Find the distance between A(−3, 2) and B(5, −4). Then find the distance between (1, 2) and (4, 7), "
                "as a root or to two decimal places.",
        "answer": [10, sp.sqrt(34)], "tol": [1e-9, 0.006],
        "moves": ["across 5 − (−3) = 8, up −4 − 2 = −6; then Pythagoras.",
                  "across 3, up 5: the square root of 9 + 25."],
        "working": ["A to B: across 5 − (−3) = 8, up −4 − 2 = −6.",
                    "d = √(8² + (−6)²) = √(64 + 36) = √100 = 10.",
                    "(1, 2) to (4, 7): across 3, up 5, so d = √(9 + 25) = √34 ≈ 5.83."]}
assert dist((-3, 2), (5, -4)) == 10 and round(dist((1, 2), (4, 7)), 2) == 5.83
practice(2)

## 3 · The section formula

The point that divides AB internally in the ratio m : n is ((mx₂ + nx₁)/(m + n), (my₂ + ny₁)/(m + n)): each end is
weighted by the other end's number. The midpoint is the 1 : 1 case, ((x₁ + x₂)/2, (y₁ + y₂)/2). The centroid of a
triangle, where its medians meet, is the average of its three corners.

In [ ]:
def section(P, R, m, n): return (F(m * R[0] + n * P[0], m + n), F(m * R[1] + n * P[1], m + n))
CEN3 = ((2, -1), (6, 4), (-5, 3))

Q[3] = {"text": "Find the point that divides A(−2, 3) to B(8, −2) internally in the ratio 3 : 2 (two coordinates). Then "
                "find the centroid of the triangle with corners (2, −1), (6, 4) and (−5, 3) (two more).",
        "answer": [*section((-2, 3), (8, -2), 3, 2), F(sum(p[0] for p in CEN3), 3), F(sum(p[1] for p in CEN3), 3)],
        "moves": ["x = (3 × 8 + 2 × (−2)) ÷ 5: B gets the 3, A gets the 2.",
                  "y = (3 × (−2) + 2 × 3) ÷ 5.",
                  "the centroid's x is the average of the three x values.",
                  "the centroid's y is the average of the three y values."],
        "working": ["x = (3 × 8 + 2 × (−2)) ÷ (3 + 2) = (24 − 4) ÷ 5 = 4.",
                    "y = (3 × (−2) + 2 × 3) ÷ 5 = (−6 + 6) ÷ 5 = 0. So the point is (4, 0).",
                    "Check: (4, 0) is 3/5 of the way from A, since 4 − (−2) = 6 is 3/5 of 10.",
                    "Centroid: ((2 + 6 − 5) ÷ 3, (−1 + 4 + 3) ÷ 3) = (3 ÷ 3, 6 ÷ 3) = (1, 2)."]}
assert Q[3]["answer"] == [4, 0, 1, 2]
practice(3)

## 4 · Area of a triangle

For corners (x₁, y₁), (x₂, y₂), (x₃, y₃), the area is ½|x₁(y₂ − y₃) + x₂(y₃ − y₁) + x₃(y₁ − y₂)|. If the bracket is
zero, there's no triangle at all: the three points are collinear, on one straight line.

In [ ]:
def bracket(P, R, S): return P[0] * (R[1] - S[1]) + R[0] * (S[1] - P[1]) + S[0] * (P[1] - R[1])
K4 = sp.solve(bracket((1, 4), (3, sp.Symbol("k")), (7, 10)), sp.Symbol("k"))[0]

Q[4] = {"text": "Find the area of the triangle with corners (2, 1), (6, 3) and (3, 7). Then find k so that (1, 4), "
                "(3, k) and (7, 10) lie on one straight line.",
        "answer": [F(abs(bracket((2, 1), (6, 3), (3, 7))), 2), K4],
        "moves": ["½|2(3 − 7) + 6(7 − 1) + 3(1 − 3)|.",
                  "collinear means the area is zero: set the bracket for (1, 4), (3, k), (7, 10) equal to 0."],
        "working": ["Bracket: 2(3 − 7) + 6(7 − 1) + 3(1 − 3) = −8 + 36 − 6 = 22, so the area is ½ × 22 = 11.",
                    "For the line: 1(k − 10) + 3(10 − 4) + 7(4 − k) = k − 10 + 18 + 28 − 7k = 36 − 6k.",
                    "36 − 6k = 0 gives k = 6.",
                    "Check with slopes: (6 − 4) ÷ (3 − 1) = 1 and (10 − 6) ÷ (7 − 3) = 1. Same slope, one line."]}
assert Q[4]["answer"] == [11, 6] and F(6 - 4, 3 - 1) == F(10 - 6, 7 - 3)
practice(4)

## 5 · Slope of a line

Slope is rise over run: m = (y₂ − y₁)/(x₂ − x₁). It's positive uphill, negative downhill, zero for a flat line, and a
vertical line has no slope. If the line makes an angle θ with the positive x axis (its inclination), then m = tan θ.

In [ ]:
Q[5] = {"text": "Find the slope of the line through (−1, 4) and (3, −4). Then give the slope of a line whose angle of "
                "inclination is 120°.",
        "answer": [F(-4 - 4, 3 - (-1)), -sp.sqrt(3)],
        "moves": ["rise −4 − 4 = −8 over run 3 − (−1) = 4.",
                  "m = tan 120°, and 120° = 180° − 60°, so tan 120° = −tan 60°."],
        "working": ["Rise: −4 − 4 = −8. Run: 3 − (−1) = 4. Slope = −8 ÷ 4 = −2 (downhill).",
                    "m = tan 120° = −tan 60° = −√3 ≈ −1.732. An obtuse inclination gives a negative slope."]}
assert Q[5]["answer"][0] == -2 and abs(math.tan(math.radians(120)) + math.sqrt(3)) < 1e-12
practice(5)

## 6 · Parallel and perpendicular lines, and the angle between two lines

Parallel lines have the same slope: m₁ = m₂. Perpendicular lines have slopes that multiply to −1: m₁m₂ = −1, so you
flip the slope and change its sign. The acute angle θ between two lines has tan θ = |(m₂ − m₁)/(1 + m₁m₂)|.

In [ ]:
def tan_between(m1, m2): return abs(F(m2 - m1) / (1 + F(m1) * m2))

Q[6] = {"text": "A line passes through (1, 2) and (4, 8). Give the slope of every line perpendicular to it. Then find "
                "the acute angle between y = 3x + 1 and y = −2x + 4, in degrees.",
        "answer": [F(-1, 2), 45],
        "moves": ["its slope is (8 − 2) ÷ (4 − 1) = 2; flip it and change the sign.",
                  "tan θ = |(−2 − 3) ÷ (1 + 3 × (−2))|."],
        "working": ["Slope of the line: (8 − 2) ÷ (4 − 1) = 6 ÷ 3 = 2.",
                    "Perpendicular: 2 × m = −1, so m = −1/2.",
                    "tan θ = |(−2 − 3) ÷ (1 + 3 × (−2))| = |−5 ÷ −5| = 1, so θ = 45°."]}
assert F(8 - 2, 4 - 1) * F(-1, 2) == -1 and tan_between(3, -2) == 1
practice(6)

## 7 · Equations of a straight line

One line, many ways to write it. **Point-slope**: y − y₁ = m(x − x₁). **Slope-intercept**: y = mx + c, where c is
where it crosses the y axis. **Intercept form**: x/a + y/b = 1, crossing the axes at (a, 0) and (0, b).
**General**: Ax + By + C = 0. Move the four sliders to move the two points: the line through them shows its slope,
its intercepts and its equation in all four forms at once.

In [ ]:
def coef(m):
    """m in front of a letter: 2, −1, (2/3) … with 1 and −1 left as just the sign."""
    m = F(m)
    if m == 1: return ""
    if m == -1: return "−"
    if m.denominator == 1: return num(m)
    return f"({num(m)})" if m > 0 else f"−({num(-m)})"

def plus(v, lead=" + "):
    """+ 3 or − 3 after a term (nothing for zero)."""
    v = fr(v)
    return "" if v == 0 else (f"{lead}{num(v)}" if v > 0 else f" − {num(-v)}")

def under(v):
    """A number under a fraction bar: 4 stays 4; −1/2 and 1/3 get brackets."""
    v = F(v)
    return num(v) if v.denominator == 1 and v > 0 else f"({num(v)})"

def shift(letter, v):
    """x − 2, x + 3, or just x."""
    return letter if v == 0 else f"{letter}{plus(-fr(v))}"

def general(P, R):
    """A, B, C of Ax + By + C = 0 through P and R: whole numbers, no common factor, A positive (or B if A is 0)."""
    A, B = R[1] - P[1], P[0] - R[0]
    C = -(A * P[0] + B * P[1])
    g = math.gcd(math.gcd(A, B), C) or 1
    s = -1 if A < 0 or (A == 0 and B < 0) else 1
    return s * A // g, s * B // g, s * C // g

def line_forms(P, R):
    """Slope, the two intercepts and the four forms of the line through two different points P and R."""
    A, B, C = general(P, R)
    gen = (f"{coef(A)}x" if A else "") + ((f" + {coef(B)}y" if B > 0 else f" − {coef(-B)}y") if A and B else
                                          (f"{coef(B)}y" if B else "")) + plus(C) + " = 0"
    if P[0] == R[0]:                                         # upright: x = k
        return {"slope": "none (an upright line)", "x-int": num(P[0]),
                "y-int": "every point (the y axis)" if P[0] == 0 else "none",
                "point-slope": "none: no slope", "slope-intercept": "none: no slope",
                "intercept": f"x/{num(P[0])} = 1" if P[0] else "none: through the origin", "general": gen,
                "m": None, "c": None, "xi": F(P[0])}
    m = F(R[1] - P[1], R[0] - P[0]); c = P[1] - m * P[0]
    mx = f"{coef(m)}x" if m else ""
    si = f"y = {mx}{plus(c) if mx else num(c)}"
    ps = (f"{shift('y', P[1])} = 0" if m == 0 else
          f"{shift('y', P[1])} = {coef(m)}" + (f"({shift('x', P[0])})" if P[0] else "x"))
    xi = None if m == 0 else -c / m
    icpt = ("none: through the origin" if c == 0 else "none: it never meets the x axis" if m == 0 else
            f"x/{under(xi)} + y/{under(c)} = 1")
    return {"slope": num(m), "x-int": "none" if xi is None else num(xi), "y-int": num(c), "point-slope": ps,
            "slope-intercept": si, "intercept": icpt, "general": gen, "m": m, "c": c, "xi": xi}

_f = line_forms((-2, -1), (4, 3))                            # the opening position: slope 2/3
assert (_f["m"], _f["c"], _f["x-int"], general((-2, -1), (4, 3))) == (F(2, 3), F(1, 3), "−1/2", (2, -3, 1))
assert _f["slope-intercept"] == "y = (2/3)x + 1/3" and _f["point-slope"] == "y + 1 = (2/3)(x + 2)"
assert _f["intercept"] == "x/(−1/2) + y/(1/3) = 1" and _f["general"] == "2x − 3y + 1 = 0"
assert line_forms((0, 0), (3, -2))["intercept"] == "none: through the origin"
assert line_forms((-3, 5), (2, 5))["point-slope"] == "y − 5 = 0" and line_forms((2, -1), (2, 4))["general"] == "x − 2 = 0"
assert line_forms((1, 2), (4, 8))["point-slope"] == "y − 2 = 2(x − 1)"
assert line_forms((0, 3), (3, 0))["intercept"] == "x/3 + y/3 = 1"
for P_, R_ in (((1, 2), (4, 8)), ((-3, 5), (2, 5)), ((2, -1), (2, 4)), ((0, 0), (3, -2)), ((-1, 4), (3, -4))):
    A_, B_, C_ = general(P_, R_)
    assert A_ * P_[0] + B_ * P_[1] + C_ == 0 == A_ * R_[0] + B_ * R_[1] + C_      # both points on the general form

def line_cell(x1=-2, y1=-1, x2=4, y2=3):
    fig = plt.figure(figsize=(11.8, 5.0))
    ax = fig.add_axes([0.0, 0.06, 0.46, 0.88]); grid(ax)
    p = fig.add_axes([0.47, 0.0, 0.53, 1.0]); p.axis("off")
    if (x1, y1) == (x2, y2):
        ax.plot([x1], [y1], "o", color=CORAL, ms=13)
        p.text(0.02, 0.6, "Both points are in the same place.\nMove one: a line needs two different points.",
               color=GOLD, fontsize=16, va="top", linespacing=1.6)
        plt.show(); return
    f = line_forms((x1, y1), (x2, y2))
    L = 6.5
    if f["m"] is None: ax.plot([x1, x1], [-L, L], color=CHALK, lw=2.6)
    else:
        xs = np.array([-L, L]); ax.plot(xs, float(f["m"]) * xs + float(f["c"]), color=CHALK, lw=2.6)
    if f["m"] not in (None, 0):                                              # rise over run, from the first point
        ax.plot([x1, x2], [y1, y1], color=SKY, lw=3, ls="--"); ax.plot([x2, x2], [y1, y2], color=MINT, lw=3, ls="--")
    if f["m"] is not None and abs(float(f["c"])) <= L: ax.plot([0], [float(f["c"])], "s", color=GOLD, ms=10, zorder=5)
    if f["xi"] is not None and abs(float(f["xi"])) <= L: ax.plot([float(f["xi"])], [0], "s", color=GOLD, ms=10, zorder=5)
    for name, (px, py) in (("A", (x1, y1)), ("B", (x2, y2))):
        ax.plot([px], [py], "o", color=CORAL, ms=13, zorder=6)
        ax.text(px + 0.3, py + 0.35, f"{name}({num(px)}, {num(py)})", color=CORAL, fontsize=13, fontweight="bold")
    p.text(0.0, 0.94, f"slope {f['slope']}", color=CHALK, fontsize=17, fontweight="bold")
    p.text(0.0, 0.85, f"crosses the x axis at {f['x-int']}  ·  the y axis at {f['y-int']}", color=GOLD, fontsize=13.5)
    rows = (("point-slope", f["point-slope"]), ("slope-intercept", f["slope-intercept"]),
            ("intercept form", f["intercept"]), ("general", f["general"]))
    for k, (name, eq) in enumerate(rows):
        p.text(0.0, 0.70 - 0.165 * k, name, color=DIM, fontsize=13)
        p.text(0.0, 0.63 - 0.165 * k, eq, color=CORAL if k == 3 else CHALK, fontsize=17, family="monospace")
    plt.show()

In [ ]:
interact(line_cell, x1=IntSlider(value=-2, min=-6, max=6, description="A: x"),
         y1=IntSlider(value=-1, min=-6, max=6, description="A: y"),
         x2=IntSlider(value=4, min=-6, max=6, description="B: x"),
         y2=IntSlider(value=3, min=-6, max=6, description="B: y"));

In [ ]:
Q[7] = {"text": "Find the equation of the line through (2, −1) that is perpendicular to 3x − 4y + 5 = 0. Type it as "
                "an equation, in any form.",
        "answer": [4 * x + 3 * y - 5],
        "working": ["Slope of 3x − 4y + 5 = 0: rearrange to y = (3/4)x + 5/4, so m = 3/4.",
                    "Perpendicular slope: flip and change the sign, −4/3.",
                    "Point-slope through (2, −1): y + 1 = −(4/3)(x − 2).",
                    "Multiply by 3: 3y + 3 = −4x + 8, so 4x + 3y − 5 = 0 (or y = −(4/3)x + 5/3).",
                    "Check: 4 × 2 + 3 × (−1) − 5 = 0, and the slopes 3/4 and −4/3 multiply to −1."]}
assert 4 * 2 + 3 * -1 - 5 == 0 and F(3, 4) * F(-4, 3) == -1
assert same_curve(read("y = -4/3x + 5/3")[0], Q[7]["answer"][0]) and not same_curve(read("4x - 3y = 5")[0], Q[7]["answer"][0])
practice(7)

## 8 · Distance of a point from a line

The shortest way from a point to a line meets the line at a right angle. From (x₁, y₁) to Ax + By + C = 0 it is
d = |Ax₁ + By₁ + C| / √(A² + B²). Two parallel lines Ax + By + C₁ = 0 and Ax + By + C₂ = 0 (the same A and B) are
|C₁ − C₂| / √(A² + B²) apart.

In [ ]:
def pt_line(P, A, B, C):
    """|Ax₁ + By₁ + C| / √(A² + B²), exact when the root is a whole number."""
    top, bottom = abs(A * P[0] + B * P[1] + C), A * A + B * B
    return F(top, math.isqrt(bottom)) if math.isqrt(bottom) ** 2 == bottom else top / math.sqrt(bottom)

Q[8] = {"text": "Find the distance of the point (3, −2) from the line 5x − 12y − 9 = 0. Then find the distance between "
                "the parallel lines 3x + 4y − 7 = 0 and 6x + 8y + 1 = 0. Fractions or two decimal places.",
        "answer": [pt_line((3, -2), 5, -12, -9), F(abs(-7 - F(1, 2)), 5)], "tol": [0.006, 0.006],
        "moves": ["|5 × 3 − 12 × (−2) − 9| ÷ √(25 + 144).",
                  "first make the x and y numbers match: halve 6x + 8y + 1 = 0 to 3x + 4y + 1/2 = 0."],
        "working": ["|5 × 3 − 12 × (−2) − 9| = |15 + 24 − 9| = 30, and √(25 + 144) = √169 = 13.",
                    "So d = 30/13 ≈ 2.31.",
                    "Halve the second line: 3x + 4y + 1/2 = 0. Now both are 3x + 4y + C = 0.",
                    "Gap = |−7 − 1/2| ÷ √(9 + 16) = 7.5 ÷ 5 = 3/2 = 1.5."]}
assert Q[8]["answer"] == [F(30, 13), F(3, 2)]
practice(8)

## 9 · The circle

A circle with centre (h, k) and radius r is (x − h)² + (y − k)² = r². Multiply it out and you get the general form
x² + y² + 2gx + 2fy + c = 0, with centre (−g, −f) and radius √(g² + f² − c). An equation is a circle only when x²
and y² have the same number in front, there's no xy term, and g² + f² − c is positive. Type any equation into the
box and press Enter: it draws the circle and marks its centre and radius, or tells you why it isn't one.

In [ ]:
def circle_of(text):
    """Read an equation and decide: a circle (centre, radius), a single point, nothing, or not a circle."""
    e = sp.expand(read_one(prep(text)))
    if not e.free_symbols <= {x, y}: return {"kind": "no", "why": "use only the letters x and y."}
    P = sp.Poly(e, x, y)
    if P.total_degree() != 2: return {"kind": "no", "why": "a circle needs x² and y²: this has no squared terms."
                                      if P.total_degree() < 2 else "it has powers higher than 2."}
    A, C, B = P.coeff_monomial(x ** 2), P.coeff_monomial(y ** 2), P.coeff_monomial(x * y)
    if B != 0: return {"kind": "no", "why": "it has an xy term, and a circle never does."}
    if A == 0 or C == 0: return {"kind": "no", "why": "it needs both x² and y² (with only one, it's a parabola or "
                                                    "a pair of lines)."}
    if A != C: return {"kind": "no", "why": f"x² and y² must have the same number in front. Here it's {num(A)} and {num(C)}: "
                                            f"{'an ellipse, or no points' if A * C > 0 else 'a hyperbola'}."}
    g, f, c = P.coeff_monomial(x) / (2 * A), P.coeff_monomial(y) / (2 * A), P.coeff_monomial(1) / A
    rr = g ** 2 + f ** 2 - c
    if rr < 0: return {"kind": "none", "centre": (-g, -f), "rr": rr, "g": g, "f": f, "c": c}
    return {"kind": "point" if rr == 0 else "circle", "centre": (-g, -f), "r": sp.sqrt(rr), "rr": rr,
            "g": g, "f": f, "c": c}

assert circle_of("x^2 + y^2 - 4x + 6y - 12 = 0")["centre"] == (2, -3) and circle_of("x^2+y^2-4x+6y-12=0")["r"] == 5
assert circle_of("x^2 + y^2 + 8x + 10y - 8 = 0")["r"] == 7                 # the video's example: centre (−4, −5)
assert circle_of("(x - 2)^2 + (y - 6)^2 = 25")["centre"] == (2, 6)          # standard form works too
assert circle_of("2x^2 + 2y^2 - 8x = 0")["r"] == 2                          # divided through first
assert circle_of("x^2 + y^2 - 2x + 4y + 9 = 0")["kind"] == "none"           # 1 + 4 − 9 = −4: no points
assert circle_of("x^2 + 2y^2 - 4 = 0")["kind"] == "no" and circle_of("x^2 + y^2 + xy = 1")["kind"] == "no"

def circle_cell(equation="x^2 + y^2 - 4x + 6y - 12 = 0"):
    fig = plt.figure(figsize=(11.8, 5.0))
    ax = fig.add_axes([0.0, 0.06, 0.46, 0.88]); grid(ax, 8)
    p = fig.add_axes([0.48, 0.0, 0.52, 1.0]); p.axis("off")
    p.text(0.0, 0.92, equation.strip(), color=CHALK, fontsize=16, family="monospace")
    try: c = circle_of(equation)
    except Exception:
        p.text(0.0, 0.70, "I can't read that equation yet.\nWrite it like  x^2 + y^2 - 4x + 6y - 12 = 0",
               color=GOLD, fontsize=15, va="top", linespacing=1.6); plt.show(); return
    if c["kind"] == "no":
        try:                                                  # draw whatever curve it is, faintly
            X, Y = np.meshgrid(np.linspace(-8.5, 8.5, 401), np.linspace(-8.5, 8.5, 401))
            fn = sp.lambdify((x, y), read_one(prep(equation)), "numpy")
            ax.contour(X, Y, fn(X, Y) + 0 * X, levels=[0], colors=[DIM], linewidths=2)
        except Exception: pass
        p.text(0.0, 0.74, "Not a circle:", color=CORAL, fontsize=22, fontweight="bold")
        p.text(0.0, 0.64, textwrap.fill(c["why"], 44), color=CHALK, fontsize=15, va="top", linespacing=1.5)
        plt.show(); return
    h, k = c["centre"]
    p.text(0.0, 0.78, f"g = {num(c['g'])},  f = {num(c['f'])},  c = {num(c['c'])}", color=DIM, fontsize=15)
    p.text(0.0, 0.66, f"g² + f² − c = {num(c['rr'])}", color=DIM, fontsize=15)
    p.text(0.0, 0.46, f"centre ({num(h)}, {num(k)})", color=CORAL, fontsize=24, fontweight="bold")
    if c["kind"] == "none":
        p.text(0.0, 0.32, "No points at all: g² + f² − c\nis negative, so there's no radius.", color=GOLD,
               fontsize=16, va="top", linespacing=1.5)
    elif c["kind"] == "point":
        ax.plot([float(h)], [float(k)], "o", color=CORAL, ms=12)
        p.text(0.0, 0.32, "radius 0: just one point", color=GOLD, fontsize=20)
    else:
        rad = float(c["r"]); t = np.linspace(0, 2 * math.pi, 300)
        ax.plot(float(h) + rad * np.cos(t), float(k) + rad * np.sin(t), color=CHALK, lw=2.8)
        ax.plot([float(h), float(h) + rad], [float(k), float(k)], color=MINT, lw=3)
        ax.text(float(h) + rad / 2, float(k) + 0.35, "r", color=MINT, fontsize=16, ha="center")
        ax.plot([float(h)], [float(k)], "o", color=CORAL, ms=12, zorder=6)
        rs = num(c["r"]) if c["r"].is_rational else f"√{num(c['rr'])} ≈ {rad:.2f}"
        p.text(0.0, 0.32, f"radius {rs}", color=GOLD, fontsize=24, fontweight="bold")
        p.text(0.0, 0.18, f"({shift('x', h)})² + ({shift('y', k)})² = {num(c['rr'])}", color=CHALK, fontsize=16,
               family="monospace")
    plt.show()

In [ ]:
interact(circle_cell, equation=Text(value="x^2 + y^2 - 4x + 6y - 12 = 0", continuous_update=False,
                                    description="equation:", layout=Layout(width="520px")));

In [ ]:
T9 = ((2, 6), (3, -1), (-4, -2))
_g, _f, _c = sp.symbols("g f c")
S9 = sp.solve([px ** 2 + py ** 2 + 2 * _g * px + 2 * _f * py + _c for px, py in T9], [_g, _f, _c])
C9 = (-S9[_g], -S9[_f]); R9 = sp.sqrt(S9[_g] ** 2 + S9[_f] ** 2 - S9[_c])

Q[9] = {"text": "Find the circle through (2, 6), (3, −1) and (−4, −2). Type its centre (two coordinates) and its "
                "radius.",
        "answer": [*C9, R9],
        "moves": ["put each point into x² + y² + 2gx + 2fy + c = 0 and subtract the equations in pairs to lose c.",
                  "the same three equations: once you have g and f, the centre is (−g, −f).",
                  "radius = √(g² + f² − c), or the distance from the centre to any of the three points."],
        "working": ["Put each point into x² + y² + 2gx + 2fy + c = 0:",
                    "(2, 6): 40 + 4g + 12f + c = 0.   (3, −1): 10 + 6g − 2f + c = 0.   (−4, −2): 20 − 8g − 4f + c = 0.",
                    "First minus second: 30 − 2g + 14f = 0.   Second minus third: −10 + 14g + 2f = 0.",
                    "Solve these two: g = 1 and f = −2. Then c = −10 − 6 − 4 = −20.",
                    "Centre (−g, −f) = (−1, 2). Radius √(1 + 4 + 20) = √25 = 5.",
                    "Check: (2, 6) is 3 across and 4 up from (−1, 2), and √(9 + 16) = 5."]}
assert Q[9]["answer"] == [-1, 2, 5] and all(dist((-1, 2), P) == 5 for P in T9)
practice(9)

## 10 · A line and a circle

Compare d, the distance from the centre to the line, with the radius r. If d < r the line cuts the circle at two
points (a secant), if d = r it touches at one (a tangent), and if d > r it misses. When it cuts, the chord between
the two points is 2√(r² − d²) long.

In [ ]:
Q[10] = {"text": "The circle x² + y² = 25 and the line 3x + 4y = 15. Give the distance from the centre to the line, "
                 "the number of points where they meet, and the length of the chord.",
         "answer": [3, 2, 8],
         "moves": ["the centre is (0, 0): |3 × 0 + 4 × 0 − 15| ÷ √(9 + 16).",
                   "compare that distance with the radius, 5.",
                   "half the chord, the drop and the radius make a right triangle: half = √(r² − d²)."],
         "working": ["Centre (0, 0), radius 5. Write the line as 3x + 4y − 15 = 0.",
                     "d = |0 + 0 − 15| ÷ √(9 + 16) = 15 ÷ 5 = 3.",
                     "3 < 5, so the line cuts the circle at 2 points.",
                     "Half the chord = √(25 − 9) = 4, so the chord is 8.",
                     "Check: the points are (5, 0) and (−1.4, 4.8), and √(6.4² + 4.8²) = √64 = 8."]}
_ends = sp.solve([x ** 2 + y ** 2 - 25, 3 * x + 4 * y - 15], [x, y])
assert len(_ends) == 2 and sp.sqrt((_ends[0][0] - _ends[1][0]) ** 2 + (_ends[0][1] - _ends[1][1]) ** 2) == 8
assert pt_line((0, 0), 3, 4, -15) == 3
practice(10)

## 11 · Tangents to a circle

A tangent meets the radius at a right angle. For x² + y² = r², the tangent at the point (x₁, y₁) is xx₁ + yy₁ = r².
From a point P outside the circle, the two tangents are the same length: √(PC² − r²), where C is the centre.

In [ ]:
Q[11] = {"text": "Find the tangent to x² + y² = 25 at the point (−3, 4), as an equation in any form. Then find the "
                 "length of a tangent from (7, 5) to the same circle.",
         "answer": [-3 * x + 4 * y - 25, sp.sqrt(7 ** 2 + 5 ** 2 - 25)],
         "moves": ["xx₁ + yy₁ = r² with (x₁, y₁) = (−3, 4).",
                   "√(PC² − r²): PC² = 7² + 5², and r² = 25."],
         "working": ["Tangent at (−3, 4): x(−3) + y(4) = 25, so −3x + 4y = 25 (or y = (3/4)x + 25/4).",
                     "Check: the radius to (−3, 4) has slope −4/3, the tangent has slope 3/4, and they multiply to −1.",
                     "Tangent length from (7, 5): √(49 + 25 − 25) = √49 = 7."]}
assert -3 * -3 + 4 * 4 == 25 and F(-4, 3) * F(3, 4) == -1 and Q[11]["answer"][1] == 7
practice(11)

## Sources
NCERT Mathematics Class X, ch. 7 (Coordinate Geometry); Class XI, ch. 9 (Straight Lines) and ch. 10 (Conic Sections:
the circle); OpenStax, College Algebra 2e, ch. 2.1–2.4 and 8.1. All retrieved 2026-10-31. Every question is our own.